# Surrogate-1 v2 — Colab T4 (full v18 technique stack)

**Free GPU**: Google Colab T4 16GB · ~12-15h session.

**This notebook runs `bin/surrogate-1-train-v18-mission.py` (2900+ lines, R1-R12 + V13-V18 stack)** — ported from the original Kaggle T4×2 target. Hardware-aware base-model picker auto-selects Qwen2.5-Coder-7B for 16GB single-GPU (T4×2 32GB would auto-pick 14B; H100 80GB → 32B).

## What's in v18 (vs the prior baseline)

| Round | Technique | Status on Colab T4 |
|---|---|---|
| R1 | LoRA r=32 + all-linear (q/k/v/o/gate/up/down) | ✓ |
| R2 | DoRA decomposition (`use_dora=True`) | ✓ |
| R3 | Liger Kernel | ⚠ skipped (T4 SM 7.5; needs SM 8.0+) |
| R4 | Flash Attention 2 | ⚠ skipped (T4 SM 7.5; FA2 needs SM 8.0+) |
| R5 | Sample packing (`packing=True`) — **4-8× throughput** | ✓ |
| R6 | NEFTune noise α=5 | ✓ |
| R7 | YaRN context (RoPE config in adapter) | ✓ serve-time |
| R8 | Gradient checkpointing | ✓ |
| R9 | AdamW 8-bit paged | ✓ |
| R10 | BF16 if SM≥8, FP16 else | T4 → FP16 |
| R11 | Cosine LR + 3% warmup | ✓ |
| R12 | Multi-source interleave (axentx-pairs A+B+C+D) | ✓ |
| V13 | TRL ≥0.21 (AsyncGRPO, GKDTrainer, DPO improvements) | ✓ |
| V13 | PEFT ≥0.19 (LoRA-GA initialization) | ✓ |
| V13 | APOLLO-Mini optimizer alt path | ✓ opt-in |
| V14 | UNIFIED INGEST+TRAIN (Phase -1: distill via Cerebras/Groq → push 9 datasets, then training pulls them) | ✓ |
| V15 | LoRA init via PiSSA / LoftQ / CorDA | ✓ via `SUR_LORA_INIT` env |
| V16 | LoRA+ (different LR for A/B matrices) | ✓ via `SUR_LORA_PLUS_RATIO` |
| V17 | Spectrum analysis: only fine-tune top-N% most-changed layers | ✓ via `SPECTRUM_TOP_FRACTION` |
| V18 | Hardware-vs-base sanity (auto-drop FP8 on T4) + adapter-naming-by-base-family | ✓ |
| Optional | GRPO post-SFT pass (reasoning RL) | ✓ via `RUN_GRPO=1` |

## Run order

1. **Cell 1** (env + secrets) — set `HF_TOKEN` in Colab secrets panel first
2. **Cell 2** (fetch v18 script from GitHub)
3. **Cell 3** (run training — 6-10h on T4 for 7B + 50k samples)
4. Cell 6 of the v18 script auto-pushes to `axentx/surrogate-1-{SIZE}B-v1.5-...`

## Resume on disconnect

Colab kills sessions on idle. The v18 script auto-checkpoints every 500 steps to `/content/v2-out/checkpoint-*`. To resume, just re-run Cell 3 — `SFTTrainer` picks up from the latest checkpoint.

In [ ]:
# Cell 1 — env + secrets (Colab Secrets panel → 🔑 → add HF_TOKEN, etc.)
import os
from google.colab import userdata

# Mandatory
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']  # alias for hf libs

# Optional
for k in ('WANDB_API_KEY', 'CEREBRAS_API_KEY', 'GROQ_API_KEY',
         'BASE_MODEL', 'HUB_MODEL_ID', 'SUR_LORA_INIT', 'SUR_LORA_PLUS_RATIO',
         'LORA_R', 'MAX_SAMPLES', 'EPOCHS', 'SEQ_LEN', 'LEARNING_RATE',
         'RUN_GRPO', 'MAGPIE_TAKE', 'TAKE_TOOLACE', 'TAKE_MULTIIAC',
         'TAKE_XLAM', 'TAKE_ITBENCH', 'TAKE_CODEFB',
         'DISABLE_AL', 'AL_SAMPLE_CAP', 'SPECTRUM_TOP_FRACTION'):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass

# Safe defaults for Colab T4 16GB (override via Colab Secrets if needed)
os.environ.setdefault('BASE_MODEL', 'Qwen/Qwen2.5-Coder-7B-Instruct')
os.environ.setdefault('LORA_R', '32')              # R1 — was 16 in baseline
os.environ.setdefault('MAX_SAMPLES', '50000')      # T4 12-15h budget
os.environ.setdefault('EPOCHS', '1')
os.environ.setdefault('SEQ_LEN', '2048')
os.environ.setdefault('LEARNING_RATE', '2e-4')
os.environ.setdefault('SUR_LORA_INIT', 'pissa_niter_4')   # V15 — better init
os.environ.setdefault('SUR_LORA_PLUS_RATIO', '16.0')      # V16 — LoRA+
os.environ.setdefault('SPECTRUM_TOP_FRACTION', '0.5')     # V17 — top-50% layers
os.environ.setdefault('RUN_GRPO', '0')                    # post-SFT RL — opt-in

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv
import torch
print(f'cuda: {torch.cuda.is_available()}  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'compute capability: {torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "none"}')
print(f'\nv18 env summary:')
for k in ('BASE_MODEL','LORA_R','MAX_SAMPLES','EPOCHS','SEQ_LEN','LEARNING_RATE',
         'SUR_LORA_INIT','SUR_LORA_PLUS_RATIO','SPECTRUM_TOP_FRACTION','RUN_GRPO'):
    print(f'  {k}={os.environ.get(k,"-")}')


In [ ]:
# Cell 2 — fetch v18 script from GitHub (always-fresh)
#
# We don't pin the script to a release tag because v18 is the active
# tip — bug fixes for new HF transformers / TRL releases land here
# regularly. If you want a frozen version, set V18_REF to a commit SHA
# in Colab Secrets and the curl below honors it.
import os
REF = userdata.get('V18_REF') if False else 'main'  # default to main
URL = f'https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/bin/surrogate-1-train-v18-mission.py'
!curl -sSf -o /content/v18_train.py {URL!s}
!wc -l /content/v18_train.py
!head -30 /content/v18_train.py


In [ ]:
# Cell 3 — run v18 training (6-10h, all techniques active)
#
# v18's deps install is idempotent (subprocess pip install --quiet),
# bootstraps from env (Kaggle Secrets path is gracefully skipped on Colab),
# auto-picks 7B for 16GB T4, runs Phase -1 distillation if Cerebras/Groq
# tokens present, then SFT, then optional GRPO if RUN_GRPO=1.
#
# Cell 6 of the v18 script auto-pushes to:
#   axentx/surrogate-1-{SIZE}B-v1.5-{base-family}
# (Override with HUB_MODEL_ID env above.)
%cd /content
%run /content/v18_train.py


In [ ]:
# Cell 4 — sanity check the pushed adapter (optional)
from huggingface_hub import HfApi
import os
api = HfApi(token=os.environ['HF_TOKEN'])
# v18 names the repo by detected size; auto-discover the most-recent push:
user = api.whoami()['name']
models = api.list_models(author='axentx', filter='peft', sort='lastModified', direction=-1, limit=3)
print('Most-recent axentx peft pushes:')
for m in models:
    print(f'  {m.id}  ({m.lastModified})')


In [ ]:
# Cell 5 — deploy the HF Space v2 (one-click)
#
# After Cell 3 finishes, the LoRA is on Hub. This cell pulls the deploy
# script + runs it. Targets ashirato/surrogate-1-v2 by default (PRO +
# ZeroGPU eligible). Set SPACE_ID in Colab Secrets to override.
import os
REF = 'main'
!curl -sSf -o /content/deploy.sh https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/hf-space-v2/deploy.sh
!mkdir -p /content/hf-space-v2 && cd /content/hf-space-v2 && \
  curl -sSf -O https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/hf-space-v2/app.py && \
  curl -sSf -O https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/hf-space-v2/requirements.txt && \
  curl -sSf -O https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/hf-space-v2/README.md && \
  curl -sSf -O https://raw.githubusercontent.com/arkashira/surrogate-1-harvest/{REF}/hf-space-v2/.gitattributes
os.environ.setdefault('SPACE_ID', 'ashirato/surrogate-1-v2')
!cd /content/hf-space-v2 && bash /content/deploy.sh
